# Céleste 레이어 분리 (See-through)

원화 한 장을 **앞머리·뒷머리·얼굴·눈·옷** 같은 부위별 레이어(PSD)로 나눕니다. 가려진 부분도 채워 줍니다.

**쓰는 법**
1. 위 메뉴 **런타임 → 런타임 유형 변경 → T4 GPU** 선택 후 저장
2. **런타임 → 모두 실행** (처음엔 설치·모델 내려받기로 10~15분)
3. 마지막 셀이 끝나면 `celeste_layers.zip` 이 자동으로 내려받아집니다 → 그 파일을 Claude에게 올려 주세요

중간에 빨간 오류가 나면 그 화면을 캡처해서 보내 주세요.

In [ ]:
# 1) GPU 확인 — 'Tesla T4' 같은 이름이 나와야 합니다
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 2) See-through 코드 받기
import os, shutil
os.chdir('/content')                                   # 항상 바깥 폴더에서 시작
shutil.rmtree('/content/st', ignore_errors=True)
!git clone -q --depth 1 https://huggingface.co/spaces/24yearsold/see-through-demo /content/st
!ln -sf common/assets /content/st/assets
print('코드 받기 성공' if os.path.isdir('/content/st/common') else '코드 받기 실패')

In [ ]:
# 3) 필요한 라이브러리 설치 (Colab에 있는 torch는 그대로 씀)
#    빨간 글씨로 "pip's dependency resolver does not currently take into account..." 가 나와도 정상입니다.
#    맨 아래에 버전 목록이 찍히면 설치 성공 → 다음 셀로 넘어가세요.
import subprocess, sys
def pip(*pk):
    return subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-U', *pk]).returncode == 0
# See-through가 맞춰 둔 torch 2.8 (CUDA 12.8)로 맞춘다. Colab 기본(최신 CUDA 13)은 T4에서 계산 라이브러리가 시작되지 않을 수 있음
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torch==2.8.0', 'torchvision==0.23.0',
                '--index-url', 'https://download.pytorch.org/whl/cu128'])
common = ['safetensors', 'einops', 'pycocotools', 'scikit-learn', 'opencv-python-headless', 'pyyaml']
pip(*common)
# See-through가 맞춰 둔 버전으로 고정 (최신 버전은 내부 사용법이 달라 실패할 수 있음)
if not pip('psd-tools[composite]==1.14.2'): pip('psd-tools[composite]')
if not pip('bitsandbytes==0.49.2'): pip('bitsandbytes')
if not pip('accelerate==1.13.0'): pip('accelerate')
# See-through가 쓰는 버전을 먼저 시도하고, 안 되면 최신 버전으로
if not pip('diffusers==0.37.0'): pip('diffusers')
if not pip('transformers==5.0.0', 'huggingface-hub==1.7.2'): pip('transformers', 'huggingface-hub')
import importlib.metadata as md
for n in ['torch', 'diffusers', 'transformers', 'huggingface-hub', 'accelerate', 'bitsandbytes', 'psd-tools']:
    try: print(f'{n:16s}', md.version(n))
    except Exception as e: print(f'{n:16s} 설치 안 됨  ← 이 줄을 캡처해서 보내 주세요')

In [ ]:
# 4) 입력 그림 받기 (Céleste 상반신, 정사각형)
!wget -q -O /content/input.png https://raw.githubusercontent.com/vecar0/midnight/celeste-concept/work/celeste/seethrough_input.png
from PIL import Image; im = Image.open('/content/input.png'); print(im.size); im.resize((384,384))

In [ ]:
# 5) 레이어 분리 실행 — GPU를 먼저 검사하고, 되는 설정을 차례로 자동 시도합니다
import os, subprocess, shutil, glob

def fresh_code():
    os.chdir('/content')
    shutil.rmtree('/content/st', ignore_errors=True)
    g = subprocess.run(['git', 'clone', '-q', '--depth', '1', 'https://huggingface.co/spaces/24yearsold/see-through-demo', '/content/st'],
                       capture_output=True, text=True)
    assert os.path.isdir('/content/st/common'), 'See-through 코드 받기 실패: ' + g.stderr
    subprocess.run('ln -sf common/assets assets', shell=True, cwd='/content/st')

def gpu_mem():
    return subprocess.run(['nvidia-smi', '--query-gpu=name,memory.used,memory.total', '--format=csv,noheader'],
                          capture_output=True, text=True).stdout.strip()

env = dict(os.environ, PYTHONPATH='/content/st/common:/content/st', PYTORCH_CUDA_ALLOC_CONF='expandable_segments:True')

# (1) GPU 기본 계산 테스트
t = subprocess.run(['python', '-c', 'import torch; print(torch.__version__, torch.cuda.get_device_name(0)); '
                    'a = torch.randn(512, 512, device="cuda"); print("행렬 곱셈 OK", float((a @ a).sum()) != 0); '
                    'b = a.half(); print("float16 OK", float((b @ b).float().sum()) != 0)'],
                   capture_output=True, text=True, env=env)
print('[GPU 검사]', gpu_mem()); print(t.stdout.strip()); 
if t.returncode != 0:
    print(t.stderr[-1500:])
    raise SystemExit('GPU 기본 계산부터 실패 → 런타임 → 런타임 연결 해제 및 삭제 후, T4 GPU로 다시 모두 실행해 주세요')

# (2) 설정을 차례로 시도
tries = [
    # T4는 bfloat16에서 메모리 절약형 attention을 못 써서 수십 GB를 요구한다 → float16을 먼저
    ('4비트, float16, 1024',        ['--resolution', '1024', '--no_group_offload'], True),
    ('4비트, float16, 896',         ['--resolution', '896',  '--no_group_offload'], True),
    ('4비트, float16, 768',         ['--resolution', '768',  '--no_group_offload'], True),
]
OK = False
for name, extra, fp16 in tries:
    fresh_code(); shutil.rmtree('/content/out', ignore_errors=True)
    if fp16:
        subprocess.run('grep -rl "bfloat16" common inference/scripts | xargs sed -i "s/torch.bfloat16/torch.float16/g"', shell=True, cwd='/content/st')
    print(f'\n=== 시도: {name} ===  GPU {gpu_mem()}')
    p = subprocess.Popen(['python', '-u', 'inference/scripts/inference_psd_quantized.py', '--srcp', '/content/input.png',
                          '--save_dir', '/content/out', '--save_to_psd', '--tblr_split', *extra],
                         cwd='/content/st', env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    tail = []
    for line in p.stdout:                     # 진행 막대가 실시간으로 보인다
        print(line, end=''); tail = (tail + [line])[-4:]
    p.wait()
    if os.path.exists('/content/out/input.psd'):
        print('성공!'); OK = True; break
    print('실패 (종료 코드', p.returncode, ')'); print(''.join(tail))
if not OK:
    print('\n모든 설정이 실패했습니다 — 이 셀 아래 글자를 전부 복사해서 보내 주세요')

In [ ]:
# 7) 결과 미리보기 + 내려받기
import glob, shutil, matplotlib.pyplot as plt
pngs = [p for p in sorted(glob.glob('/content/out/input/*.png')) if not p.endswith('_depth.png')]
print(len(pngs), '개 레이어')
cols = 6; rows = (len(pngs)+cols-1)//cols
plt.figure(figsize=(18, 3*rows))
for i, p in enumerate(pngs):
    plt.subplot(rows, cols, i+1); plt.imshow(Image.open(p)); plt.title(os.path.basename(p)[:-4], fontsize=8); plt.axis('off')
plt.show()
shutil.make_archive('/content/celeste_layers', 'zip', '/content/out')
from google.colab import files; files.download('/content/celeste_layers.zip')